In [1]:
!pip install pandas openpyxl numpy

In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set random seed to make data reproducible
np.random.seed(42)
random.seed(42)

# Config for synthetic data
num_records = 7500
client_types = ['Retail', 'Wholesale', 'Online', 'VIP', 'Corporate']
products = ['T-Shirt', 'Jeans', 'Jacket', 'Sweater', 'Sneakers', 'Socks', 'Dress', 'Skirt']
sizes = ['XS', 'S', 'M', 'L', 'XL', 'XXL']
colors = ['Red', 'Blue', 'Black', 'White', 'Green', 'Grey', 'Yellow']

# Generate basic data
data = []
start_date = datetime(2022, 1, 1)

for i in range(num_records):
    client_type = random.choice(client_types)
    # Generate random dates over a ~2 year period
    date = start_date + timedelta(days=random.randint(0, 700), hours=random.randint(0, 23), minutes=random.randint(0, 59))
    
    product = random.choice(products)
    size = random.choice(sizes)
    color = random.choice(colors)
    
    # Introduce messiness: inconsistent capitalization
    if random.random() < 0.1:
        product = product.lower()
    elif random.random() < 0.05:
        product = product.upper()
        
    # Introduce messiness: missing values (NaN)
    if random.random() < 0.03:
        size = np.nan
    if random.random() < 0.02:
        color = None
        
    # Different buying behavior per client type
    if client_type in ['Wholesale', 'Corporate']:
        quantity = random.randint(10, 100)
    else:
        quantity = random.randint(1, 5)
        
    # Baseline prices
    price_base = {'T-Shirt': 20, 'Jeans': 50, 'Jacket': 120, 'Sweater': 60, 'Sneakers': 80, 'Socks': 10, 'Dress': 70, 'Skirt': 40}
    product_clean = product.title() if isinstance(product, str) else product
    price = price_base.get(product_clean, 50) * (1 + random.uniform(-0.15, 0.15))
    
    # Introduce messiness: mixed date formats (some as string, some as datetime object)
    if random.random() < 0.05:
        date_val = date.strftime('%d/%m/%Y %H:%M')
    else:
        date_val = date
        
    data.append([
        f"ORD-{10000+i}",
        date_val,
        client_type,
        f"Client_{random.randint(1, 300)}",
        product,
        size,
        color,
        round(price, 2),
        quantity,
        round(price * quantity, 2)
    ])

# Convert to DataFrame
df = pd.DataFrame(data, columns=['OrderID', 'Date', 'ClientType', 'ClientID', 'Product', 'Size', 'Color', 'UnitPrice', 'Quantity', 'TotalAmount'])

# Introduce one more element of messiness: duplicate rows
duplicates = df.sample(n=150, random_state=1)
df = pd.concat([df, duplicates], ignore_index=True)
# Shuffle the dataframe
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save to Excel with different tabs per client type
file_name = 'messy_clothing_sales.xlsx'

print(f"Generating Excel file '{file_name}'...")
with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
    for c_type in client_types:
        df_type = df[df['ClientType'] == c_type].copy()
        
        # Drop the ClientType col since it's implied by the sheet name
        df_type.drop('ClientType', axis=1, inplace=True) 
        
        # Add tab-specific messiness: shuffle column order for some tabs
        if c_type == 'Wholesale':
            cols = ['Date', 'OrderID', 'ClientID', 'Product', 'Quantity', 'TotalAmount', 'UnitPrice', 'Size', 'Color']
            df_type = df_type[cols]
        elif c_type == 'Corporate':
            # Drop an empty column just to be inconsistent
            df_type['Notes'] = np.nan
            
        df_type.to_excel(writer, sheet_name=c_type, index=False)

print(f"Success! '{file_name}' generated with {len(df)} total rows across {len(client_types)} sheets.")

Generating Excel file 'messy_clothing_sales.xlsx'...


Success! 'messy_clothing_sales.xlsx' generated with 7650 total rows across 5 sheets.
